In [1]:
pip install transformers datasets torch pandas scikit-learn

  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ---------------------------------------- 11.5/11.5 MB 72.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   --------------------------------------- 555.1/555.1 kB 18.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/765.1 kB ? eta -:--:--
   --------------------------------------- 765.1/765.1 kB 33.5 MB/s eta 0:00:00
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
   ---------------------------------------- 0.0/27.4 MB ? eta -:--:--
   --------------------------- ------------ 18.9/27.4 MB 85.0 MB/s eta 0:00:01
   ---------------------------------------- 27.4/27.4 MB 72.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 80.2 MB/s eta 0:00:00
   -

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
qiskit-nature 0.8.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.35.1 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [4]:
import os
# Mute the Windows Symlink warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Suppress the unauthenticated request warning (unless you need private models)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1" 

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
from sklearn.metrics import accuracy_score, classification_report

# Only show actual errors from transformers, hiding the "missing keys" printout
logging.set_verbosity_error()

# 1. Load your local CSV files
train_df = pd.read_csv(r"C:\Users\Lenovo\Desktop\Downloads\Phishing URL dataset\train_bert_represented.csv")
test_df = pd.read_csv(r"C:\Users\Lenovo\Desktop\Downloads\Phishing URL dataset\test_bert_represented.csv")

# 2. Initialize the ELECTRA Tokenizer
model_name = "google/electra-small-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Create a PyTorch Dataset class
class PhishingDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['URL'].astype(str).tolist()  
        self.labels = df['Label'].tolist()          
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Instantiate PyTorch Datasets and DataLoaders
train_dataset = PhishingDataset(train_df, tokenizer)
test_dataset = PhishingDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 4. Initialize ELECTRA for Sequence Classification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# 5. Native PyTorch AdamW Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# 6. Training Loop
epochs = 3
print("Starting training...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

# 7. Evaluation Loop
model.eval()
predictions = []
true_labels = []

print("\nStarting evaluation...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).flatten()
        
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# 8. Print Final Metrics
print("\n--- Evaluation Metrics ---")
print(f"Accuracy: {accuracy_score(true_labels, predictions):.4f}\n")
print(classification_report(true_labels, predictions))

Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Starting training...
Epoch 1/3 | Loss: 0.0785
Epoch 2/3 | Loss: 0.0199
Epoch 3/3 | Loss: 0.0119

Starting evaluation...

--- Evaluation Metrics ---
Accuracy: 0.9959

              precision    recall  f1-score   support

           0       0.99      1.00      1.00     14995
           1       1.00      0.99      1.00     10961

    accuracy                           1.00     25956
   macro avg       1.00      1.00      1.00     25956
weighted avg       1.00      1.00      1.00     25956



In [6]:
pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


   ---------------------------------------- 0.0/874.7 kB ? eta -:--:--
   --------------------------------------- 874.7/874.7 kB 40.8 MB/s eta 0:00:00


In [8]:
pip install sentencepiece protobuf

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 16.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [24]:
%pip install sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import io

# Disable UI widgets, progress bars, and telemetry alerts permanently
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1" 
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "True"

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
from sklearn.metrics import accuracy_score, classification_report

# Only show actual errors from transformers
logging.set_verbosity_error()

# 1. Load your local CSV files using standard text streams to bypass Windows file locks
train_path = r"C:\Users\Lenovo\Desktop\Downloads\Phishing URL dataset\train_bert_represented.csv"
test_path = r"C:\Users\Lenovo\Desktop\Downloads\Phishing URL dataset\test_bert_represented.csv"

with open(train_path, "r", encoding="utf-8") as f:
    train_df = pd.read_csv(io.StringIO(f.read()))
with open(test_path, "r", encoding="utf-8") as f:
    test_df = pd.read_csv(io.StringIO(f.read()))

# 2. Initialize the RoBERTa Tokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Create a PyTorch Dataset class
class PhishingDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['URL'].astype(str).tolist()  
        self.labels = df['Label'].tolist()          
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Instantiate PyTorch Datasets and DataLoaders
train_dataset = PhishingDataset(train_df, tokenizer)
test_dataset = PhishingDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 4. Initialize RoBERTa for Sequence Classification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# 5. Native PyTorch AdamW Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# 6. Training Loop
epochs = 3
print("Starting training...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

# 7. Evaluation Loop
model.eval()
predictions = []
true_labels = []

print("\nStarting evaluation...")
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).flatten()
        
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# 8. Print Final Metrics
print("\n--- Evaluation Metrics ---")
print(f"Accuracy: {accuracy_score(true_labels, predictions):.4f}\n")
print(classification_report(true_labels, predictions))